# Thêm Thư Viện

In [1]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [2]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2025;'
)
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=Library_DWH;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2025;'
)

## Đọc data từ SQL Server

In [3]:
query_Lop = "SELECT Ten_lop FROM Lop" # Đọc dữ liệu từ bảng Lop trong CSDL libol
df_lop_lop = pd.read_sql(query_Lop, conn_libol)
query_LopBandoc = "SELECT DISTINCT dbo.DecodeUTF8String(Lop) AS Lop FROM Ban_doc" # Đọc dữ liệu từ bảng Ban_doc trong CSDL libol
df_lop_bandoc = pd.read_sql(query_LopBandoc, conn_libol)
print(df_lop_lop)
print(df_lop_bandoc)

C:\Users\admin\AppData\Local\Temp\ipykernel_1012\675338709.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_lop_lop = pd.read_sql(query_Lop, conn_libol)
C:\Users\admin\AppData\Local\Temp\ipykernel_1012\675338709.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_lop_bandoc = pd.read_sql(query_LopBandoc, conn_libol)


       Ten_lop
0      191040B
1      191040A
2    19109CL1B
3    19109CL1A
4    19109CL2B
..         ...
597    211611B
598    211611A
599    211612A
600    211612B
601      21950

[602 rows x 1 columns]
            Lop
0        017031
1       001011C
2       042030A
3       057090A
4       117450A
...         ...
4257        709
4258   14151CLC
4259    181311A
4260  19143CL1B
4261      23950

[4262 rows x 1 columns]


## Xử lý data

In [4]:
# Tạo data_frame mới gộp các hàng dữ liệu từ 2 data_frame kia
df_data_lop = pd.DataFrame({"ID_lop": pd.concat([df_lop_lop["Ten_lop"], 
                                                  df_lop_bandoc["Lop"]], 
                                                  ignore_index=True)})
df_data_lop['ID_lop'] = df_data_lop['ID_lop'].str.upper() # In hoa hết các hàng dữ liệu
for j, row in df_data_lop.iterrows():
    ten_nhom = row["ID_lop"]
    if ((pd.isna(ten_nhom)) or # Kiểm tra none
        (ten_nhom == "") or
        (ten_nhom == "0") or
        (ten_nhom == "00") or
        (ten_nhom == "000")):  # Kiểm tra NaN
        df_data_lop.at[j, 'ID_lop'] = "0"
df_data_lop = df_data_lop.sort_values(by="ID_lop", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn
df_data_lop = df_data_lop.drop_duplicates().reset_index(drop=True) # xóa những hàng bị trùng nhau
df_data_lop['ID_khoa'] = "0" # cho ID_Khoa = 0 (Không rõ) vì hiện tại chưa có dữ liệu về lớp thuộc khoa nào 
print(df_data_lop)

        ID_lop ID_khoa
0            0       0
1       001011       0
2      001011A       0
3      001011C       0
4       001012       0
...        ...     ...
4254  XÂY DỰNG       0
4255   ÊN11021       0
4256  ÊN14010A       0
4257  ÊN2D02VD       0
4258      ĐIỆN       0

[4259 rows x 2 columns]


## Load data

### [Nếu cần] Clear bảng

In [5]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM olap.DIM_Lop"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load data vào bảng Dim

In [6]:
cursor_dwh = conn_dwh_library.cursor()
insert_query = """
                INSERT INTO olap.DIM_Lop (ID_lop, ID_khoa) 
                VALUES (?, ?)
                """
for index, row in df_data_lop.iterrows():
    values = (row['ID_lop'], 
              row['ID_khoa'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_library.commit()